# N1 solution. Guided activity of Notebook 1, solved

**Predicting Flash Floods in Working Landscapes**, Module 3, Event based evaluation and forecast skill
CUAHSI Water Science Exchange 2026

This short notebook holds the completed activity of Notebook 1 (section 6), with its outputs. It rebuilds only what the activity needs, so it runs on its own in a few seconds. The full notebook with every section executed is `01_model_and_control_run_solution.ipynb`.

The task. Sweep a second parameter, the split `Alp`, over the same flood event and report how far the simulated peak moves.

## 0. Setup

In [1]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/AHWALab/Water-Science-Exchange-Workshop.git"
BRANCH = "module-3-evaluation"

# Step 1. If the helper files are not next to this notebook, we are in Colab. Copy the branch.
if not Path("hymod_tools.py").exists():
    if not Path("Water-Science-Exchange-Workshop").is_dir():
        subprocess.run(["git", "clone", "--quiet", "--depth", "1", "-b", BRANCH, REPO_URL, "Water-Science-Exchange-Workshop"], check=True)
    os.chdir("Water-Science-Exchange-Workshop/notebooks")

# Step 2. Make the helper files importable.
sys.path.insert(0, os.getcwd())
print("Working folder:", os.getcwd())

Working folder: /home/claude/m3/notebooks


## 1. What the activity needs from Notebook 1

The data, the evaluation window, the calibrated control parameters `h.CONTROL_PARS` and the event slice of section 5.

In [2]:
import numpy as np
import hymod_tools as h

PP, PET, QQ = h.load_leaf_river("LeafCatch.dat")      # 40 years of daily data, mm per day
N_EVAL = 365 * 5                                        # the five year evaluation window
Qobs = QQ[:N_EVAL]
event = slice(450, 560)                                 # the flood event of section 5
print("control parameters", h.CONTROL_PARS)

control parameters {'Huz': 171.3386, 'B': 0.9021, 'Alp': 0.5679, 'Nq': 2, 'Kq': 0.6429, 'Ks': 0.0559}


## 2. The activity, solved

In [3]:
#################### Your Code Goes Here #########################
# Step 1. Eight values of Alp between 0.2 and 0.9 (np.linspace).
alp_values = np.linspace(0.2, 0.9, 8)

# Step 2. For each value, copy h.CONTROL_PARS, replace "Alp", run the model
#         over the evaluation window, and store the peak of Q inside the event slice.
peaks = []
for alp in alp_values:
    pars = dict(h.CONTROL_PARS); pars["Alp"] = alp
    Q = h.hymod01(PP[:N_EVAL], PET[:N_EVAL], pars)["Q"]
    peaks.append(Q[event].max())

# Step 3. Print the smallest and largest simulated peak, and the observed peak of the event.
print(f"simulated peak from {min(peaks):.2f} to {max(peaks):.2f} mm/day, observed {Qobs[event].max():.2f} mm/day")
###################################################################

simulated peak from 18.56 to 48.20 mm/day, observed 45.94 mm/day


Reading the result. `Alp` splits the effective rainfall between the quick and the slow path, so it moves the peak less than `Kq`, which sets how fast the quick path drains. Compare the two ranges with the ones printed in section 5 of Notebook 1.